In [ ]:
import cloudscraper
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, BooleanType, TimestampType
import time
import io
from pyspark.sql import functions as F
from datetime import datetime
from bs4 import BeautifulSoup
import warnings
import os
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

warnings.filterwarnings('ignore')
%load_ext sparksql_magic


builder = SparkSession.builder \
    .appName("DeltaLakeLocal") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# Automatically downloads the correct matching Delta jars
spark = configure_spark_with_delta_pip(builder).getOrCreate()
%config SparkSqlMagic.spark = spark

your 131072x1 screen size is bogus. expect trouble
26/05/22 19:57:52 WARN Utils: Your hostname, DESKTOP-IS86RQE resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/22 19:57:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/obaliuta/miniconda3/envs/spark_with_delta/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/obaliuta/.ivy2/cache
The jars for the packages stored in: /home/obaliuta/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2b9bae4e-2f63-422a-8de1-49fcd7ff0f8f;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 560ms :: artifacts dl 28ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |

In [4]:

def get_gas_prices_by_state(state_code):

    url = f"https://gasprices.aaa.com/?state={state_code.upper()}"
    
    scraper = cloudscraper.create_scraper()
    headers = {
        'User-Agent': 'Mozilla/5.0 (iPhone; CPU iPhone OS 14_7_1 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.2 Mobile/15E148 Safari/604.1',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
    }
    
    try:
        response = scraper.get(url, headers=headers, timeout=30)
        
        if response.status_code != 200:
            print(f"Failed for {state_code}. Code: {response.status_code}")
            return None
        
        soup = BeautifulSoup(response.content, 'html.parser')
        tables = soup.find_all('table')
        
        if not tables:
            print(f"No tables found for {state_code}")
            return None
        
        if len(tables) < 3:
            print(f"Not enough tables found for {state_code} (found {len(tables)}, need at least 3)")
            return None
        
        # Skip first 2 tables, start from 3rd table (index 2)
        tables = tables[2:]
        
        all_data = []
        
        for idx, table in enumerate(tables):
            try:
                df = pd.read_html(io.StringIO(str(table)))[0]
                
                # Find the h3 tag that appears BEFORE this table
                city_name = None
                
                # Look for the closest h3 tag before this table
                h3_tag = table.find_previous('h3')
                if h3_tag:
                    city_name = h3_tag.get_text(strip=True)
                
                # Fallback to default if not found
                if not city_name or city_name == '':
                    city_name = f"Region_{idx+1}"
                    
                df['State'] = state_code.upper()
                df['City/Region'] = city_name
                df['Scrape_Date'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                
                all_data.append(df)
                
            except Exception as e:
                print(f"  Error processing table {idx+1}: {str(e)}")
        
        if all_data:
            combined_df = pd.concat(all_data, ignore_index=True)
            return combined_df
        else:
            return None
            
    except Exception as e:
        print(f"Error fetching data for {state_code}: {str(e)}")
        return None


In [5]:
def get_gas_prices_multiple_states(state_codes, delay=2):
    # Handle single string input
    if isinstance(state_codes, str):
        state_codes = [state_codes]
    
    all_states_data = []
    
    for state in state_codes:
        df = get_gas_prices_by_state(state)
        
        if df is not None:
            all_states_data.append(df)
        else:
            print(f"Failed to retrieve data for {state.upper()}\n")
        
        # Delay between requests
        if state != state_codes[-1]:
            time.sleep(delay)
    
    if all_states_data:
        combined_df = pd.concat(all_states_data, ignore_index=True)
        return combined_df
    else:
        return None

In [6]:
states = ['WA', 'OR', 'CA', 'TX', 'NY', 'FL', 'IL', 'PA', 'OH', 'MI', 'GA', 'NC', 'NJ', 'VA', 'MA', 'AZ', 'IN', 'TN', 'MO', 'MD', 'WI', 'MN', 'CO', 'AL', 'SC', 'LA', 'KY', 'OR', 'OK', 'CT', 'UT', 'IA', 'NV', 'AR', 'MS', 'KS', 'NE', 'WV', 'ID', 'HI', 'NH', 'ME', 'RI', 'DE', 'VT']

data = get_gas_prices_multiple_states(states)

data = data.rename(columns={'Unnamed: 0': "Measurement"})
data = data[['State', 'City/Region', 'Measurement', 'Regular', 'Mid', 'Premium', 'Diesel', 'Scrape_Date']]
data = data[data['Regular'].notna()]


for c in ['Regular', 'Mid', 'Premium', 'Diesel']:
    data[c] = data[c].str.replace('$', '').str.strip()
    data[c] = pd.to_numeric(data[c], errors='coerce')   

In [7]:
spark_df = spark.createDataFrame(data)

In [25]:
# Save DataFrame as a Delta Table using the absolute home path
home_dir = os.path.expanduser("~")
delta_path = os.path.join(home_dir, "lakehouse_test/bronze/gas_prices")

spark_df.write.format("delta").mode("append").save(delta_path)
print(f"Delta table created successfully at: {delta_path}")

26/05/22 19:31:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Delta table created successfully at: /home/obaliuta/lakehouse_test/bronze/gas_prices


In [8]:
todays_data = (spark_df.filter(F.col("Measurement") == "Current Avg.").drop("Measurement")
                .withColumnRenamed("Scrape_Date", "Date").withColumn("Date", F.col("Date").cast("date"))
                .selectExpr(
                    "State", "`City/Region`", "Date",
                    "stack(4, 'Regular', Regular, 'Mid', Mid, 'Premium', Premium, 'Diesel', Diesel) as (Fuel_Type, Price)"
                )
).withColumnRenamed("City/Region", "City_Region")

todays_data_tx = todays_data.filter(F.col("State") == "TX")

In [9]:
# enrichment with the geodata

geolocator = Nominatim(user_agent="gas_prices_geocoder", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Get distinct State/City_Region pairs to minimise API calls
locations_pd = todays_data_tx.select("State", "City_Region").distinct().toPandas()

def get_geodata(row):
    location = geocode(f"{row['City_Region']}, {row['State']}, USA")
    if location:
        return pd.Series({
            'latitude': location.latitude,
            'longitude': location.longitude,
            'full_address': location.address,
            'country': location.raw.get('display_name', '').split(',')[-1].strip(),
        })
    return pd.Series({'latitude': None, 'longitude': None, 'full_address': None, 'country': None})


In [10]:
geo_pd = locations_pd.copy()
geo_pd[['latitude', 'longitude', 'full_address', 'country']] = locations_pd.apply(get_geodata, axis=1)

geo_spark = spark.createDataFrame(geo_pd)
todays_data_tx_geo = todays_data_tx.join(geo_spark, on=["State", "City_Region"], how="left")

In [13]:
todays_data_tx_geo.show(5, truncate=False)

+-----+-----------------+----------+---------+-----+----------+------------+--------------------------------------------------------------------------+-------------+
|State|City_Region      |Date      |Fuel_Type|Price|latitude  |longitude   |full_address                                                              |country      |
+-----+-----------------+----------+---------+-----+----------+------------+--------------------------------------------------------------------------+-------------+
|TX   |Amarillo         |2026-05-22|Regular  |4.008|35.20729  |-101.8371192|Amarillo, Potter County, Texas, United States                             |United States|
|TX   |Amarillo         |2026-05-22|Mid      |4.355|35.20729  |-101.8371192|Amarillo, Potter County, Texas, United States                             |United States|
|TX   |Amarillo         |2026-05-22|Premium  |4.758|35.20729  |-101.8371192|Amarillo, Potter County, Texas, United States                             |United States|
|TX 

In [14]:
# Save DataFrame as a Delta Table using the absolute home path
home_dir = os.path.expanduser("~")
delta_path = os.path.join(home_dir, "lakehouse_test/silver/gas_prices_tx_daily")

todays_data_tx_geo.write.format("delta").mode("append").save(delta_path)
print(f"Delta table created successfully at: {delta_path}")

26/05/22 20:06:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Delta table created successfully at: /home/obaliuta/lakehouse_test/silver/gas_prices_tx_daily


In [ ]:
spark.stop()